# CMAPSS – Robustness & XAI unter KI‑Störungen (RUL‑Regression)
## Neu: XGBoost auf Sequenzen (Flattened Windows) + Under/Overfitting

**Modelle**
- **LSTM (PyTorch)** auf Sequenzen
- **XGBoost (sklearn API)** auf Sequenzen: Sliding Window → **Flatten** → Regression auf RUL

**Störungen**
- Covariance Shift
- Environmental Shift
- Model Drift
- Catastrophic Forgetting
- Underfitting / Overfitting

**XAI (pro Szenario)**
- LSTM: Captum **Integrated Gradients**
- XGB-SEQ: **ICE**
- XGB-SEQ: Surrogate **Ridge**
- Regeln: **RuleFit (Regression)** + Surrogate **DecisionTreeRegressor**
- Counterfactuals: **DiCE desired_range** (wenn verfügbar) sonst **Wachter CF** (scipy)


In [1]:
# --------------------------
# 0) Imports / Setup
# --------------------------
import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())

import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor, export_text

import xgboost as xgb

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Captum (IG)
HAS_CAPTUM = True
try:
    from captum.attr import IntegratedGradients
except Exception as e:
    HAS_CAPTUM = False
    print("Captum/IntegratedGradients nicht verfügbar:", repr(e))

# DiCE
HAS_DICE = True
try:
    import dice_ml
    from dice_ml import Dice
except Exception as e:
    HAS_DICE = False
    print("dice-ml nicht verfügbar:", repr(e))

# RuleFit
HAS_RULEFIT = True
try:
    from rulefit import RuleFit
except Exception as e:
    HAS_RULEFIT = False
    print("rulefit nicht verfügbar:", repr(e))

# Wachter CF (scipy)
HAS_SCIPY = True
try:
    from scipy.optimize import minimize
except Exception as e:
    HAS_SCIPY = False
    print("scipy optimize nicht verfügbar:", repr(e))


Python: 3.10.19 | packaged by conda-forge | (main, Oct 22 2025, 22:23:22) [MSC v.1944 64 bit (AMD64)]
Platform: Windows-10-10.0.19045-SP0


C:\Users\ko\Anaconda3\envs\cmaps-xai-stable\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --------------------------
# 1) Konfiguration
# --------------------------
CONFIG = {
    "data_dir": r"C:\User\OneDrive - fir.rwth-aachen.de\Diss\99_Schreibversionen\Model 4\Daten Fallstudien\CMAPPS\CMaps",      # <-- anpassen
    "dataset_id": "FD001",
    "max_rul_clip": 130,

    "seq_len": 30,



    # Eingabedimension
    "univariate_input": True,            # True => nur ein Feature (1D)
    "univariate_feature": "s_1",         # Spaltenname (z.B. "s_2"); None => erstes Sensor-Feature

     # Splits
    "train_frac": 0.7,
    "val_frac_within_train": 0.2,

    # CMAPSS-Style Störungen
    "cov_shift_scale": 1.3,
    "cov_shift_frac_features": 0.25,
    "env_shift_bias": 0.5,
    "env_shift_frac_features": 0.30,

    # Model drift
    "drift_extra_epochs": 20,   # CNN: weitertrainieren
    "rf_retrain_on_drift": True,

    # Catastrophic forgetting (Domain B = A + (cov+env shift))
    "forget_ft_epochs": 30,     # CNN fine-tune auf B
    "forget_rf_retrain": True,

    # Under/Overfitting
    "under_single_feature": None,   # z.B. "mittelwert_temp"
    "over_small_train_size": 500,
    # XGB-SEQ Strides (Punkte-Reduktion)
    "xgb_stride_train": 3,
    "xgb_stride_test": 1,

    # LSTM
    "lstm_hidden": 64,
    "lstm_layers": 2,
    "lstm_dropout": 0.2,
    "lstm_lr": 1e-3,
    "lstm_epochs": 10,
    "batch_size": 128,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    # XGB
    "xgb_params": {
        "n_estimators": 700,
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "reg_lambda": 1.0,
        "random_state": 42,
    },

    # Störungen
    "cov_shift_scale": 1.3,
    "env_shift_bias": 0.5,
    "model_drift_extra_epochs": 3,
    "forgetting_ft_epochs": 3,

    # XAI
    "xai_n_samples": 300,
    "ice_grid_points": 25,

    # Counterfactual Ziel
    "cf_delta": 20.0,
    "cf_tol": 2.5,

    "seed": 42,
}

np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
print("Device:", CONFIG["device"])

from pathlib import Path
p = Path(CONFIG["data_dir"])
print("Data dir exists:", p.exists())
if p.exists():
    print("Files (first 20):", [f.name for f in p.glob('*')][:20])


Device: cpu
Data dir exists: True
Files (first 20): ['Damage Propagation Modeling.pdf', 'readme.txt', 'RUL_FD001.txt', 'RUL_FD002.txt', 'RUL_FD003.txt', 'RUL_FD004.txt', 'test_FD001.txt', 'test_FD002.txt', 'test_FD003.txt', 'test_FD004.txt', 'train_FD001.txt', 'train_FD002.txt', 'train_FD003.txt', 'train_FD004.txt', 'x.txt']


## 2) CMAPSS laden (inkl. RUL_end Fix)

In [3]:
from pathlib import Path

def load_cmaps(data_dir: str, dataset_id: str):
    data_dir = Path(data_dir)
    train_path = data_dir / f"train_{dataset_id}.txt"
    test_path  = data_dir / f"test_{dataset_id}.txt"
    rul_path   = data_dir / f"RUL_{dataset_id}.txt"

    if not train_path.exists():
        raise FileNotFoundError(f"Train-Datei nicht gefunden: {train_path}")
    if not test_path.exists():
        raise FileNotFoundError(f"Test-Datei nicht gefunden:  {test_path}")
    if not rul_path.exists():
        raise FileNotFoundError(f"RUL-Datei nicht gefunden:   {rul_path}")

    col_names = ["id", "cycle"] + [f"os_{i}" for i in range(1,4)] + [f"s_{i}" for i in range(1,22)]

    train = pd.read_csv(train_path, sep=r"\s+", header=None)
    test  = pd.read_csv(test_path,  sep=r"\s+", header=None)
    rul   = pd.read_csv(rul_path,   sep=r"\s+", header=None, names=["RUL_end"])

    train = train.iloc[:, :len(col_names)]
    test  = test.iloc[:,  :len(col_names)]
    train.columns = col_names
    test.columns  = col_names

    # TRAIN RUL
    max_cycle = train.groupby("id")["cycle"].max().rename("max_cycle")
    train = train.join(max_cycle, on="id")
    train["RUL"] = train["max_cycle"] - train["cycle"]
    train.drop(columns=["max_cycle"], inplace=True)

    # TEST RUL (FIX)
    max_cycle_t = test.groupby("id")["cycle"].max().rename("max_cycle")
    test = test.join(max_cycle_t, on="id")

    engine_ids = np.sort(test["id"].unique())
    rul_per_id = pd.DataFrame({"id": engine_ids, "RUL_end": rul["RUL_end"].values})
    test = test.merge(rul_per_id, on="id", how="left")

    test["RUL"] = test["RUL_end"] + (test["max_cycle"] - test["cycle"])
    test.drop(columns=["max_cycle", "RUL_end"], inplace=True)

    return train, test

train_df, test_df = load_cmaps(CONFIG["data_dir"], CONFIG["dataset_id"])

if CONFIG["max_rul_clip"] is not None:
    train_df["RUL"] = train_df["RUL"].clip(upper=CONFIG["max_rul_clip"])
    test_df["RUL"]  = test_df["RUL"].clip(upper=CONFIG["max_rul_clip"])

assert train_df["RUL"].isna().sum() == 0, "NaNs in train RUL"
assert test_df["RUL"].isna().sum() == 0, "NaNs in test RUL"

SENSOR_COLS = [c for c in train_df.columns if c.startswith("s_")]
OS_COLS = [c for c in train_df.columns if c.startswith("os_")]

# --- Univariate vs. multivariate Eingabe ---
if CONFIG.get("univariate_input", False):
    cand = CONFIG.get("univariate_feature", None)
    if cand is None:
        cand = SENSOR_COLS[0] if len(SENSOR_COLS) else (OS_COLS[0] if len(OS_COLS) else None)
    if cand is None or cand not in train_df.columns:
        raise ValueError(f"univariate_feature '{cand}' nicht in DataFrame-Spalten vorhanden.")
    BASE_COLS = [cand]
else:
    BASE_COLS = OS_COLS + SENSOR_COLS

print("BASE_COLS:", BASE_COLS)

print("train:", train_df.shape, "test:", test_df.shape)
train_df.head()

BASE_COLS: ['s_1']
train: (20631, 27) test: (13096, 27)


,id,cycle,os_1,os_2,os_3,s_1,s_2,s_3,s_4,s_5,...,s_13,s_14,s_15,s_16,s_17,s_18,s_19,s_20,s_21,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,130
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,130
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,130
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,130
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,130


## 3) Datasets: LSTM-Sequenzen + XGB-Sequenzen (Flatten)

In [4]:
def make_sequences(df: pd.DataFrame, base_cols, seq_len: int):
    df = df.sort_values(["id", "cycle"])
    X_seq, y_seq = [], []
    for uid, g in df.groupby("id"):
        g = g.sort_values("cycle")
        data = g[base_cols].values.astype(np.float32)
        rul  = g["RUL"].values.astype(np.float32)
        if len(g) < seq_len:
            continue
        for i in range(len(g) - seq_len + 1):
            X_seq.append(data[i:i+seq_len])
            y_seq.append(rul[i+seq_len-1])
    return np.stack(X_seq), np.array(y_seq)

def make_xgb_sequence_dataset(df: pd.DataFrame, base_cols, seq_len: int, stride: int = 1):
    X_list, y_list = [], []
    for uid, g in df.groupby("id"):
        g = g.sort_values("cycle").reset_index(drop=True)
        data = g[base_cols].values.astype(np.float32)
        rul  = g["RUL"].values.astype(np.float32)
        if len(g) < seq_len:
            continue
        for end in range(seq_len - 1, len(g), stride):
            window = data[end - seq_len + 1 : end + 1]
            X_list.append(window.reshape(-1))
            y_list.append(rul[end])
    X = np.stack(X_list)
    y = np.array(y_list)
    feature_names = [f"{c}_t{-k}" for k in range(seq_len-1, -1, -1) for c in base_cols]
    return X, y, feature_names

# LSTM dataset
X_seq_train_full, y_seq_train_full = make_sequences(train_df, BASE_COLS, CONFIG["seq_len"])
X_seq_test_full,  y_seq_test_full  = make_sequences(test_df,  BASE_COLS, CONFIG["seq_len"])

idx = np.arange(len(y_seq_train_full))
np.random.shuffle(idx)
split = int(0.8 * len(idx))
tr_idx, va_idx = idx[:split], idx[split:]

X_seq_tr, y_seq_tr = X_seq_train_full[tr_idx], y_seq_train_full[tr_idx]
X_seq_va, y_seq_va = X_seq_train_full[va_idx], y_seq_train_full[va_idx]

seq_scaler = StandardScaler()
seq_scaler.fit(X_seq_tr.reshape(-1, X_seq_tr.shape[-1]))

def scale_seq(x):
    xf = x.reshape(-1, x.shape[-1])
    xs = seq_scaler.transform(xf)
    xs = np.nan_to_num(xs, nan=0.0, posinf=0.0, neginf=0.0)
    return xs.reshape(x.shape).astype(np.float32)

X_seq_tr_s = scale_seq(X_seq_tr)
X_seq_va_s = scale_seq(X_seq_va)
X_seq_te_s = scale_seq(X_seq_test_full)

print("LSTM seq shapes:", X_seq_tr_s.shape, X_seq_va_s.shape, X_seq_te_s.shape)

# XGB-SEQ dataset
X_xgb_train, y_xgb_train, XGB_SEQ_COLS = make_xgb_sequence_dataset(
    train_df, BASE_COLS, CONFIG["seq_len"], stride=CONFIG["xgb_stride_train"]
)
X_xgb_test,  y_xgb_test,  _ = make_xgb_sequence_dataset(
    test_df, BASE_COLS, CONFIG["seq_len"], stride=CONFIG["xgb_stride_test"]
)

X_train, X_val, y_train, y_val = train_test_split(
    X_xgb_train, y_xgb_train, test_size=0.2, random_state=CONFIG["seed"]
)

xgb_scaler = StandardScaler()
X_train_s = xgb_scaler.fit_transform(X_train)
X_val_s   = xgb_scaler.transform(X_val)
X_test_s  = xgb_scaler.transform(X_xgb_test)

print("XGB-SEQ shapes:", X_train_s.shape, X_test_s.shape)


LSTM seq shapes: (14184, 30, 1) (3547, 30, 1) (10196, 30, 1)
XGB-SEQ shapes: (4757, 30) (10196, 30)


## 4) Modelle: Training + Evaluation

In [5]:
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

class LSTMRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, num_layers=n_layers,
                            batch_first=True, dropout=dropout if n_layers > 1 else 0.0)
        self.head = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 1))
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.head(h[-1]).squeeze(-1)

def train_lstm(model, train_loader, val_loader, epochs=10, lr=1e-3, device="cpu"):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    best_val, best_state = float("inf"), None
    for ep in range(1, epochs+1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
        model.eval()
        vloss = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                vloss.append(loss_fn(model(xb), yb).item())
        v = float(np.mean(vloss))
        print(f"Epoch {ep:02d}: val={v:.4f}")
        if v < best_val:
            best_val = v
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def eval_regression(y_true, y_pred, name=""):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    return {"name": name, "RMSE": rmse, "MAE": mae, "R2": r2}

# Baseline train
xgb_clean = xgb.XGBRegressor(**CONFIG["xgb_params"])
xgb_clean.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)], verbose=False)

train_loader = DataLoader(SeqDataset(X_seq_tr_s, y_seq_tr), batch_size=CONFIG["batch_size"], shuffle=True)
val_loader   = DataLoader(SeqDataset(X_seq_va_s, y_seq_va), batch_size=CONFIG["batch_size"], shuffle=False)

lstm_clean = LSTMRegressor(len(BASE_COLS), CONFIG["lstm_hidden"], CONFIG["lstm_layers"], CONFIG["lstm_dropout"])
lstm_clean = train_lstm(lstm_clean, train_loader, val_loader, epochs=CONFIG["lstm_epochs"], lr=CONFIG["lstm_lr"], device=CONFIG["device"])

print("XGB clean:", eval_regression(y_val, xgb_clean.predict(X_val_s), "XGB-clean-val"))
with torch.no_grad():
    yhat_lstm = lstm_clean(torch.tensor(X_seq_te_s).to(CONFIG["device"])).detach().cpu().numpy()
print("XGB test:", eval_regression(y_xgb_test, xgb_clean.predict(X_test_s), "XGB-clean-test"))
print("LSTM test:", eval_regression(y_seq_test_full, yhat_lstm, "LSTM-clean-test"))


Epoch 01: val=2931.6095
Epoch 02: val=1876.5523
Epoch 03: val=1879.5909
Epoch 04: val=1877.5175
Epoch 05: val=1877.9672
Epoch 06: val=1879.3515
Epoch 07: val=1875.5566
Epoch 08: val=1875.5340
Epoch 09: val=1881.4109
Epoch 10: val=1877.5165
XGB clean: {'name': 'XGB-clean-val', 'RMSE': 43.03428099574004, 'MAE': 38.108421325683594, 'R2': -1.7881393432617188e-06}
XGB test: {'name': 'XGB-clean-test', 'RMSE': 39.894471392114596, 'MAE': 36.72736740112305, 'R2': -0.6071004867553711}
LSTM test: {'name': 'LSTM-clean-test', 'RMSE': 40.55352787424141, 'MAE': 37.341793060302734, 'R2': -0.6606377363204956}


## 5) Szenarien + Under/Overfitting + XAI Runner

In [6]:
from dataclasses import dataclass

@dataclass
class Scenario:
    name: str
    X_train_s: np.ndarray; y_train: np.ndarray
    X_val_s: np.ndarray;   y_val: np.ndarray
    X_test_s: np.ndarray;  y_test: np.ndarray
    X_seq_tr_s: np.ndarray; y_seq_tr: np.ndarray
    X_seq_va_s: np.ndarray; y_seq_va: np.ndarray
    X_seq_te_s: np.ndarray; y_seq_te: np.ndarray
    meta: dict

def make_clean_scenario(name="baseline"):
    return Scenario(
        name=name,
        X_train_s=X_train_s, y_train=y_train,
        X_val_s=X_val_s, y_val=y_val,
        X_test_s=X_test_s, y_test=y_xgb_test,
        X_seq_tr_s=X_seq_tr_s, y_seq_tr=y_seq_tr,
        X_seq_va_s=X_seq_va_s, y_seq_va=y_seq_va,
        X_seq_te_s=X_seq_te_s, y_seq_te=y_seq_test_full,
        meta={}
    )

def _copy_arrays(*arrs):
    return [np.array(a, copy=True) for a in arrs]

def apply_covariance_shift(X_train_s, X_val_s, X_test_s, scale=1.3, frac_features=0.2, seed=42):
    rng = np.random.RandomState(seed)
    Xtr, Xva, Xte = _copy_arrays(X_train_s, X_val_s, X_test_s)
    d = Xtr.shape[1]
    k = max(1, int(frac_features * d))
    idx = rng.choice(d, size=k, replace=False)
    Xtr[:, idx] *= scale; Xva[:, idx] *= scale; Xte[:, idx] *= scale
    return Xtr, Xva, Xte, {"scaled_feature_idx": idx.tolist(), "scale": scale}


def apply_covariance_shift_seq(X_seq, scale=1.3, frac_features=0.2, seed=42):
    """Multiplikativer Covariance-Shift auf Sequenzen: skaliert Subset der Features über alle Timesteps."""
    rng = np.random.RandomState(seed)
    X = np.array(X_seq, copy=True)
    f = X.shape[-1]
    k = max(1, int(frac_features * f))
    idx = rng.choice(f, size=k, replace=False)
    X[:, :, idx] *= scale
    return X, {"scaled_feature_idx": idx.tolist(), "scale": scale}

def apply_environmental_shift_seq(X_seq_scaled, bias=0.5, frac_features=0.3, seed=42):
    rng = np.random.RandomState(seed)
    X = np.array(X_seq_scaled, copy=True)
    f = X.shape[-1]
    k = max(1, int(frac_features * f))
    idx = rng.choice(f, size=k, replace=False)
    X[:, :, idx] += bias
    return X, {"biased_feature_idx": idx.tolist(), "bias": bias}

def apply_environmental_shift_xgbseq(X_flat, n_feat, seq_len, bias=0.5, frac_features=0.3, seed=42):
    rng = np.random.RandomState(seed)
    X = np.array(X_flat, copy=True).reshape(-1, seq_len, n_feat)
    k = max(1, int(frac_features * n_feat))
    idx = rng.choice(n_feat, size=k, replace=False)
    X[:, :, idx] += bias
    return X.reshape(X_flat.shape), {"biased_feature_idx": idx.tolist(), "bias": bias}

def make_covariance_shift_scenario():
    sc = make_clean_scenario("covariance_shift")
    # XGB-SEQ: multiplicative shift on flattened sequence features
    _, _, Xte, meta = apply_covariance_shift(sc.X_train_s, sc.X_val_s, sc.X_test_s,
                                            scale=CONFIG["cov_shift_scale"], seed=CONFIG["seed"])
    sc.X_test_s = Xte

    # LSTM: multiplicative shift on sequence features (per-feature across timesteps)
    Xte_seq, meta_seq = apply_covariance_shift_seq(sc.X_seq_te_s, scale=CONFIG["cov_shift_scale"], seed=CONFIG["seed"]+1)
    sc.X_seq_te_s = Xte_seq

    sc.meta = {"covariance_shift_xgbseq": meta, "covariance_shift_seq": meta_seq}
    return sc

def make_environmental_shift_scenario():
    sc = make_clean_scenario("environmental_shift")
    Xte_seq, meta_seq = apply_environmental_shift_seq(sc.X_seq_te_s, bias=CONFIG["env_shift_bias"], seed=CONFIG["seed"]+2)
    sc.X_seq_te_s = Xte_seq
    Xte_xgb, meta_xgb = apply_environmental_shift_xgbseq(sc.X_test_s, n_feat=len(BASE_COLS), seq_len=CONFIG["seq_len"],
                                                        bias=CONFIG["env_shift_bias"], seed=CONFIG["seed"]+3)
    sc.X_test_s = Xte_xgb
    sc.meta = {"env_shift_seq": meta_seq, "env_shift_xgbseq": meta_xgb}
    return sc

def train_xgb_under_over(sc: Scenario, mode: str):
    params = dict(CONFIG["xgb_params"])
    if mode == "under":
        params["max_depth"] = 2; params["n_estimators"] = 150; params["learning_rate"] = 0.05
    elif mode == "over":
        params["max_depth"] = 12; params["n_estimators"] = 2000
        params["subsample"] = 1.0; params["colsample_bytree"] = 1.0; params["reg_lambda"] = 0.0
    model = xgb.XGBRegressor(**params)
    model.fit(sc.X_train_s, sc.y_train, eval_set=[(sc.X_val_s, sc.y_val)], verbose=False)
    return model

def train_lstm_under_over(sc: Scenario, mode: str):
    if mode == "under":
        hidden, dropout, epochs = 16, 0.6, max(3, CONFIG["lstm_epochs"]//2)
    elif mode == "over":
        hidden, dropout, epochs = 128, 0.0, max(CONFIG["lstm_epochs"] + 10, 15)
    else:
        hidden, dropout, epochs = CONFIG["lstm_hidden"], CONFIG["lstm_dropout"], CONFIG["lstm_epochs"]
    model = LSTMRegressor(len(BASE_COLS), hidden, CONFIG["lstm_layers"], dropout)
    tr_loader = DataLoader(SeqDataset(sc.X_seq_tr_s, sc.y_seq_tr), batch_size=CONFIG["batch_size"], shuffle=True)
    va_loader = DataLoader(SeqDataset(sc.X_seq_va_s, sc.y_seq_va), batch_size=CONFIG["batch_size"], shuffle=False)
    return train_lstm(model, tr_loader, va_loader, epochs=epochs, lr=CONFIG["lstm_lr"], device=CONFIG["device"])

def subsample_indices(n, k, seed=42):
    rng = np.random.RandomState(seed)
    return rng.choice(n, size=min(k, n), replace=False)

# ============================================================
# 5.x) FIXED LOCAL POINTS (aus Robust_XAI_CMStyle übernommen)
#      Ziel: gleiche lokalen Punkte über alle Szenarien
# ============================================================
from typing import Sequence, List, Dict, Tuple, Any, Optional

# Feste ILOCs (kann via CONFIG überschrieben werden)
FIXED_LOCAL_ILOCS: List[int] = list(CONFIG.get("fixed_local_ilocs", [398, 1996, 728]))

def resolve_fixed_points_for_arrays(
    n: int,
    fixed_ilocs: Sequence[int],
    point_ids: Optional[np.ndarray] = None,   # optional: stabile IDs (z.B. aus df["point_id"])
    y_true: Optional[np.ndarray] = None,      # optional, nur fürs Logging
) -> Tuple[List[Dict[str, Any]], List[Tuple[int, int]]]:
    """
    Liefert:
      local_points: [{point_id, iloc, y_true, status}, ...] inkl. invalid
      pairs: [(point_id, iloc), ...] nur gültige
    """
    if point_ids is None:
        point_ids = np.arange(n, dtype=int)
    else:
        point_ids = np.asarray(point_ids).astype(int)
        assert len(point_ids) == n, "point_ids Länge passt nicht zu n"

    local_points: List[Dict[str, Any]] = []
    pairs: List[Tuple[int, int]] = []

    for ix in fixed_ilocs:
        if ix < 0 or ix >= n:
            local_points.append({"iloc": int(ix), "status": "invalid_iloc"})
            continue
        pid = int(point_ids[ix])
        yt = float(y_true[ix]) if y_true is not None else None
        local_points.append({"point_id": pid, "iloc": int(ix), "y_true": yt, "status": "ok"})
        pairs.append((pid, int(ix)))

    return local_points, pairs


def run_ig_lstm_fixed_points(
    lstm_model,
    X_seq: np.ndarray,                 # (n, T, F)
    feature_names: Sequence[str],
    fixed_ilocs: Sequence[int] = FIXED_LOCAL_ILOCS,
    point_ids: Optional[np.ndarray] = None,
    device: str = "cpu",
    seed: int = 42,
    top_k: int = 10,
    plot_heatmaps: bool = True,
    max_feat_plot: int = 25,
):
    """
    Integrated Gradients fürs LSTM:
      - lokal: pro Fixed Point Top-Features (mean(|IG|) über time)
      - global: mean(|IG|) über alle Fixed Points & time
    """
    if not HAS_CAPTUM:
        return {"status": "skipped", "reason": "captum nicht verfügbar"}

    lstm_model.eval()
    lstm_model.to(device)

    n = len(X_seq)
    local_points, pairs = resolve_fixed_points_for_arrays(
        n, fixed_ilocs, point_ids=point_ids, y_true=None
    )
    if not pairs:
        return {"status": "skipped", "reason": "keine gültigen fixed_ilocs", "local_points": local_points, "pairs": []}

    ig = IntegratedGradients(lstm_model)

    per_point: Dict[int, Any] = {}
    all_attrs = []

    for pid, ix in pairs:
        x = torch.tensor(X_seq[ix:ix+1], dtype=torch.float32).to(device)      # (1,T,F)
        baseline = torch.zeros_like(x)
        attrs = ig.attribute(x, baselines=baseline)
        attr = attrs.detach().cpu().numpy()[0]                               # (T,F)
        all_attrs.append(attr)

        mean_abs_feat = np.mean(np.abs(attr), axis=0)                        # (F,)
        top_idx = np.argsort(-mean_abs_feat)[:top_k]
        per_point[pid] = {
            "point_id": int(pid),
            "iloc": int(ix),
            "top_features": [(feature_names[i], float(mean_abs_feat[i])) for i in top_idx],
            "mean_abs_per_feature": mean_abs_feat.tolist(),
        }

        # optional: Attribution-Plot (univariat: Lineplot; multivariat: Heatmap der |IG|-Attributionen)
if plot_heatmaps:
    top_plot = np.argsort(-mean_abs_feat)[:min(max_feat_plot, len(feature_names))]
    A = np.abs(attr[:, top_plot])  # (T, top)

    if len(top_plot) == 1:
        # Univariater Fall (oder nur 1 Top-Feature): 1D-Linie über Timesteps
        plt.figure(figsize=(10, 3.0))
        plt.plot(A[:, 0])
        plt.xlabel("timestep")
        plt.ylabel("|IG|")
        plt.title(f"IG | Lineplot (Fixed Point) pid={pid}, iloc={ix} – {feature_names[top_plot[0]]}")
        plt.tight_layout()
        plt.show()
        plt.close()
    else:
        plt.figure(figsize=(min(12, 0.35 * len(top_plot) + 2), 3.2))
        plt.imshow(A.T, aspect="auto", interpolation="nearest")
        plt.yticks(range(len(top_plot)), [feature_names[i] for i in top_plot])
        plt.xlabel("timestep")
        plt.title(f"IG | Heatmap (Fixed Point) pid={pid}, iloc={ix}")
        plt.colorbar()
        plt.tight_layout()
        plt.show()
        plt.close()

    A = np.stack(all_attrs, axis=0)                                          # (n_fixed,T,F)
    mean_abs_global = np.mean(np.abs(A), axis=(0, 1))                        # (F,)
    top_idx_g = np.argsort(-mean_abs_global)[:top_k]

    return {
        "status": "ok",
        "method": "IntegratedGradients",
        # Backwards-kompatibel zu bisherigen Prints:
        "top_features": [(feature_names[i], float(mean_abs_global[i])) for i in top_idx_g],
        # Erweiterungen:
        "global_top_features": [(feature_names[i], float(mean_abs_global[i])) for i in top_idx_g],
        "local_points": local_points,
        "pairs": pairs,
        "per_point": per_point,
    }


def compute_ice_fixed_points_regression(
    predict_fn,                   # fn(X_2d)->(n,)
    X: np.ndarray,                # (n,d)
    feature_index: int,
    fixed_ilocs: Sequence[int] = FIXED_LOCAL_ILOCS,
    point_ids: Optional[np.ndarray] = None,
    grid_points: int = 25,
):
    """
    ICE-Kurven nur für die FixedLocalPoints (Regression).
    Rückgabe: dict pid -> {"grid": [...], "pred": [...]}
    """
    n, d = X.shape
    local_points, pairs = resolve_fixed_points_for_arrays(n, fixed_ilocs, point_ids=point_ids)
    if not pairs:
        return {"status": "skipped", "reason": "keine gültigen fixed_ilocs", "local_points": local_points, "pairs": []}

    col = X[:, feature_index]
    lo, hi = np.percentile(col, 5), np.percentile(col, 95)
    grid = np.linspace(lo, hi, grid_points)

    per_point = {}
    for pid, ix in pairs:
        x0 = X[ix].copy()
        preds = []
        for g in grid:
            xg = x0.copy()
            xg[feature_index] = float(g)
            preds.append(float(predict_fn(xg.reshape(1, -1))[0]))
        per_point[pid] = {"point_id": int(pid), "iloc": int(ix), "grid": grid.tolist(), "pred": preds}

    return {
        "status": "ok",
        "feature_index": int(feature_index),
        "grid": grid.tolist(),
        "local_points": local_points,
        "pairs": pairs,
        "per_point": per_point,
    }


def counterfactuals_rul_fixed_points(
    sc: Scenario,
    xgb_model,
    fixed_ilocs: Sequence[int] = FIXED_LOCAL_ILOCS,
    point_ids: Optional[np.ndarray] = None,
):
    """
    Counterfactuals nur für FixedLocalPoints.
    Nutzt die vorhandene counterfactual_rul(sc, x0, xgb_model).
    """
    X = sc.X_test_s
    n = len(X)
    local_points, pairs = resolve_fixed_points_for_arrays(n, fixed_ilocs, point_ids=point_ids)

    examples = []
    for pid, ix in pairs:
        out = counterfactual_rul(sc, X[ix], xgb_model)
        out["point_id"] = int(pid)
        out["iloc"] = int(ix)
        examples.append(out)

    return {"status": "ok", "local_points": local_points, "pairs": pairs, "examples": examples}


def run_ig_lstm(lstm_model, X_seq, feature_names, device="cpu", seed=42):
    if not HAS_CAPTUM:
        return {"status":"skipped", "reason":"captum nicht verfügbar"}
    lstm_model.eval(); lstm_model.to(device)
    idx = subsample_indices(len(X_seq), CONFIG["xai_n_samples"], seed=seed)
    X_sub = torch.tensor(X_seq[idx], dtype=torch.float32).to(device)
    ig = IntegratedGradients(lstm_model)
    attrs = ig.attribute(X_sub, baselines=torch.zeros_like(X_sub))
    attr = attrs.detach().cpu().numpy()
    mean_abs = np.mean(np.abs(attr), axis=(0,1))
    top_idx = np.argsort(-mean_abs)[:10]
    return {"status":"ok", "method":"IntegratedGradients",
            "top_features":[(feature_names[i], float(mean_abs[i])) for i in top_idx]}

def compute_ice(predict_fn, X, feature_index, grid_points=25, n_samples=200, seed=42):
    idx = subsample_indices(len(X), n_samples, seed=seed)
    Xs = np.array(X[idx], copy=True)
    col = Xs[:, feature_index]
    lo, hi = np.percentile(col, 5), np.percentile(col, 95)
    grid = np.linspace(lo, hi, grid_points)
    ice = []
    for g in grid:
        Xg = np.array(Xs, copy=True)
        Xg[:, feature_index] = g
        ice.append(predict_fn(Xg))
    return grid, np.stack(ice, axis=0)

def plot_ice(grid, ice, title):
    plt.figure(figsize=(7,4))
    for i in range(ice.shape[1]):
        plt.plot(grid, ice[:, i], alpha=0.15)
    plt.title(title)
    plt.xlabel("Feature value")
    plt.ylabel("Prediction")
    plt.show()

def train_surrogate_ridge(X_train, y_target_pred):
    model = make_pipeline(StandardScaler(with_mean=True, with_std=True), Ridge(alpha=1.0, random_state=CONFIG["seed"]))
    model.fit(X_train, y_target_pred)
    coef = model.named_steps['ridge'].coef_
    top = np.argsort(-np.abs(coef))[:10]
    return model, {"top_coefs":[(XGB_SEQ_COLS[i], float(coef[i])) for i in top]}

def extract_rules_rulefit_reg(sc: Scenario):
    if not HAS_RULEFIT:
        return {"status":"skipped", "reason":"rulefit nicht verfügbar"}
    rf = RuleFit(tree_size=4, max_rules=200, memory_par=0.01, random_state=CONFIG["seed"])
    rf.fit(sc.X_train_s, sc.y_train, feature_names=XGB_SEQ_COLS)
    rules_df = rf.get_rules()
    rules_df = rules_df[(rules_df.coef != 0) & (rules_df.type == 'rule')].sort_values('support', ascending=False)
    return {"status":"ok", "top_rules": rules_df.head(10).to_dict("records")}

def extract_rules_surrogate_tree_reg(sc: Scenario):
    reg = DecisionTreeRegressor(max_depth=3, min_samples_leaf=50, random_state=CONFIG["seed"])
    reg.fit(sc.X_train_s, sc.y_train)
    return {"status":"ok", "rules_text": export_text(reg, feature_names=XGB_SEQ_COLS)}

def wachter_counterfactual(x0, predict_fn, target_value, feature_bounds, lam=0.01, max_iter=300):
    if not HAS_SCIPY:
        return {"status":"skipped", "reason":"scipy minimize nicht verfügbar"}
    x0 = x0.astype(float)
    def objective(x):
        pred = float(predict_fn(x.reshape(1, -1))[0])
        return (pred - target_value) ** 2 + lam * np.linalg.norm(x - x0, ord=1)
    res = minimize(objective, x0, method="L-BFGS-B", bounds=feature_bounds, options={"maxiter": max_iter})
    x_cf = res.x
    return {"status": "ok" if res.success else "failed",
            "pred0": float(predict_fn(x0.reshape(1,-1))[0]),
            "target": float(target_value),
            "pred_cf": float(predict_fn(x_cf.reshape(1,-1))[0]),
            "dist_l1": float(np.linalg.norm(x_cf - x0, ord=1)),
            "x_cf": x_cf.tolist(),
            "note": "Wachter CF fallback"}

def dice_counterfactuals_regression(sc: Scenario, x0_np: np.ndarray, xgb_model, desired_low, desired_high):
    if not HAS_DICE:
        return {"status":"skipped", "reason":"dice-ml nicht verfügbar"}
    X_train_df = pd.DataFrame(sc.X_train_s, columns=XGB_SEQ_COLS)
    df_cf = X_train_df.copy(); df_cf["RUL"] = sc.y_train
    d = dice_ml.Data(dataframe=df_cf, continuous_features=XGB_SEQ_COLS, outcome_name="RUL")
    m = dice_ml.Model(model=xgb_model, backend="sklearn")
    exp = Dice(d, m, method="random")
    x0 = pd.DataFrame([x0_np], columns=XGB_SEQ_COLS)
    try:
        cf = exp.generate_counterfactuals(x0, total_CFs=3, desired_range=[float(desired_low), float(desired_high)])
        out = cf.cf_examples_list[0].final_cfs_df
        return {"status":"ok", "cfs": out.to_dict("records"), "note":"DiCE desired_range"}
    except Exception as e:
        return {"status":"error", "reason": repr(e)}

def counterfactual_rul(sc: Scenario, x0_np: np.ndarray, xgb_model):
    pred0 = float(xgb_model.predict(x0_np.reshape(1,-1))[0])
    target = max(0.0, pred0 - float(CONFIG["cf_delta"]))
    low, high = max(0.0, target - CONFIG["cf_tol"]), target + CONFIG["cf_tol"]
    if HAS_DICE:
        out = dice_counterfactuals_regression(sc, x0_np, xgb_model, low, high)
        if out.get("status") == "ok":
            out["pred0"] = pred0; out["target_range"] = [low, high]
            return out
    bounds = list(zip(sc.X_train_s.min(axis=0), sc.X_train_s.max(axis=0)))
    return wachter_counterfactual(x0_np, lambda z: xgb_model.predict(z), target, bounds)

def evaluate_models(sc: Scenario, lstm_model, xgb_model, tag=""):
    met_xgb = eval_regression(sc.y_test, xgb_model.predict(sc.X_test_s), f"XGB-{tag}")
    lstm_model.eval()
    with torch.no_grad():
        y_pred_lstm = lstm_model(torch.tensor(sc.X_seq_te_s).to(CONFIG["device"])).detach().cpu().numpy()
    met_lstm = eval_regression(sc.y_seq_te, y_pred_lstm, f"LSTM-{tag}")
    return met_xgb, met_lstm

def run_xai_for_scenario(sc: Scenario, lstm_model, xgb_model, fixed_ilocs: Sequence[int] = FIXED_LOCAL_ILOCS):
    results = {}

    # --- LSTM IG (fixed local points + global aggregation)
    results["ig_lstm"] = run_ig_lstm_fixed_points(
        lstm_model,
        sc.X_seq_te_s,
        feature_names=BASE_COLS,
        fixed_ilocs=fixed_ilocs,
        device=CONFIG["device"],
        seed=CONFIG["seed"],
        top_k=10,
        plot_heatmaps=bool(CONFIG.get("plot_ig_heatmaps", False)),
    )

    # --- XGB-SEQ Surrogate (global)
    y_bb = xgb_model.predict(sc.X_train_s)
    surrogate, sur_meta = train_surrogate_ridge(sc.X_train_s, y_bb)
    results["surrogate_regression"] = {
        "status": "ok",
        **sur_meta,
        "fidelity_R2_val": float(r2_score(xgb_model.predict(sc.X_val_s), surrogate.predict(sc.X_val_s))),
    }

    # --- ICE (nur FixedLocalPoints)
    top_feat = results["surrogate_regression"]["top_coefs"][0][0]
    j = XGB_SEQ_COLS.index(top_feat)

    ice_info = compute_ice_fixed_points_regression(
        lambda x: xgb_model.predict(x),
        sc.X_test_s,
        j,
        fixed_ilocs=fixed_ilocs,
        grid_points=int(CONFIG["ice_grid_points"]),
    )
    ice_info["feature"] = top_feat
    results["ice"] = ice_info

    if ice_info.get("status") == "ok":
        grid = np.array(ice_info["grid"])
        preds = [ice_info["per_point"][pid]["pred"] for pid, _ in ice_info["pairs"]]
        ice_mat = np.array(preds, dtype=float).T   # (len(grid), n_points)
        plot_ice(grid, ice_mat, title=f"ICE ({sc.name}) – feature: {top_feat} – XGB-SEQ (Fixed Points)")

    # --- Rules (global)
    results["rules"] = {
        "rulefit_reg": extract_rules_rulefit_reg(sc),
        "surrogate_tree_reg": extract_rules_surrogate_tree_reg(sc),
    }

    # --- Counterfactuals (nur FixedLocalPoints)
    results["counterfactuals_rul"] = counterfactuals_rul_fixed_points(
        sc, xgb_model, fixed_ilocs=fixed_ilocs
    )

    return results


def simulate_model_drift(sc: Scenario):
    # XGB drift data
    Xtr1, Xva1, Xte1, meta_cov = apply_covariance_shift(sc.X_train_s, sc.X_val_s, sc.X_test_s,
                                                       scale=CONFIG["cov_shift_scale"], seed=CONFIG["seed"]+101)
    Xtr2, meta_env_tr = apply_environmental_shift_xgbseq(Xtr1, len(BASE_COLS), CONFIG["seq_len"],
                                                        bias=CONFIG["env_shift_bias"], seed=CONFIG["seed"]+102)
    Xva2, _ = apply_environmental_shift_xgbseq(Xva1, len(BASE_COLS), CONFIG["seq_len"],
                                              bias=CONFIG["env_shift_bias"], seed=CONFIG["seed"]+103)
    Xte2, meta_env_te = apply_environmental_shift_xgbseq(Xte1, len(BASE_COLS), CONFIG["seq_len"],
                                                        bias=CONFIG["env_shift_bias"], seed=CONFIG["seed"]+104)
    xgb_drift = xgb.XGBRegressor(**CONFIG["xgb_params"])
    xgb_drift.fit(Xtr2, sc.y_train, eval_set=[(Xva2, sc.y_val)], verbose=False)

    # LSTM drift (continue training on env-shifted train sequences)
    Xseq_tr_shift, meta_seq = apply_environmental_shift_seq(sc.X_seq_tr_s, bias=CONFIG["env_shift_bias"], seed=CONFIG["seed"]+105)
    tr_loader = DataLoader(SeqDataset(Xseq_tr_shift, sc.y_seq_tr), batch_size=CONFIG["batch_size"], shuffle=True)
    va_loader = DataLoader(SeqDataset(sc.X_seq_va_s, sc.y_seq_va), batch_size=CONFIG["batch_size"], shuffle=False)
    lstm_drift = copy.deepcopy(lstm_clean)
    lstm_drift = train_lstm(lstm_drift, tr_loader, va_loader,
                            epochs=CONFIG["model_drift_extra_epochs"], lr=CONFIG["lstm_lr"], device=CONFIG["device"])

    sc_eval = make_clean_scenario("model_drift")
    sc_eval.X_test_s = Xte2
    sc_eval.meta = {"cov_shift": meta_cov, "env_shift_xgb_train": meta_env_tr, "env_shift_xgb_test": meta_env_te, "env_shift_seq_train": meta_seq}
    return sc_eval, lstm_drift, xgb_drift

def simulate_catastrophic_forgetting(sc: Scenario):
    Xtr1, Xva1, Xte1, meta_cov = apply_covariance_shift(sc.X_train_s, sc.X_val_s, sc.X_test_s,
                                                       scale=CONFIG["cov_shift_scale"], seed=CONFIG["seed"]+201)
    XtrB, meta_env_tr = apply_environmental_shift_xgbseq(Xtr1, len(BASE_COLS), CONFIG["seq_len"],
                                                        bias=CONFIG["env_shift_bias"], seed=CONFIG["seed"]+202)
    XvaB, _ = apply_environmental_shift_xgbseq(Xva1, len(BASE_COLS), CONFIG["seq_len"],
                                              bias=CONFIG["env_shift_bias"], seed=CONFIG["seed"]+203)
    XteB, meta_env_te = apply_environmental_shift_xgbseq(Xte1, len(BASE_COLS), CONFIG["seq_len"],
                                                        bias=CONFIG["env_shift_bias"], seed=CONFIG["seed"]+204)

    xgb_A = xgb_clean
    lstm_A = copy.deepcopy(lstm_clean)

    xgb_B = xgb.XGBRegressor(**CONFIG["xgb_params"])
    xgb_B.fit(XtrB, sc.y_train, eval_set=[(XvaB, sc.y_val)], verbose=False)

    # LSTM Domain B: Covariance + Environmental auf TRAIN-Sequenzen
    Xseq_tr_cov, meta_seq_cov_tr = apply_covariance_shift_seq(sc.X_seq_tr_s, scale=CONFIG["cov_shift_scale"], seed=CONFIG["seed"]+205)
    Xseq_tr_B, meta_seq_env_tr = apply_environmental_shift_seq(Xseq_tr_cov, bias=CONFIG["env_shift_bias"], seed=CONFIG["seed"]+206)
    lstm_B = copy.deepcopy(lstm_clean)
    tr_loader = DataLoader(SeqDataset(Xseq_tr_B, sc.y_seq_tr), batch_size=CONFIG["batch_size"], shuffle=True)
    va_loader = DataLoader(SeqDataset(sc.X_seq_va_s, sc.y_seq_va), batch_size=CONFIG["batch_size"], shuffle=False)
    lstm_B = train_lstm(lstm_B, tr_loader, va_loader,
                        epochs=CONFIG["forgetting_ft_epochs"], lr=CONFIG["lstm_lr"], device=CONFIG["device"])

    scB = make_clean_scenario("catastrophic_forgetting_B")
    scB.X_test_s = XteB
    # Test-Sequenzen für Domain B cov-shiften
    Xseq_te_B, meta_seq_cov_te = apply_covariance_shift_seq(scB.X_seq_te_s, scale=CONFIG["cov_shift_scale"], seed=CONFIG["seed"]+207)
    scB.X_seq_te_s = Xseq_te_B
    scB.meta = {"cov_shift_xgb": meta_cov, "env_shift_xgb_train": meta_env_tr, "env_shift_xgb_test": meta_env_te,
              "cov_shift_seq_train": meta_seq_cov_tr, "env_shift_seq_train": meta_seq_env_tr, "cov_shift_seq_test": meta_seq_cov_te}
    return (lstm_A, xgb_A), (lstm_B, xgb_B), scB

# Build scenarios
SCENARIOS = [
    make_clean_scenario("baseline"),
    make_covariance_shift_scenario(),
    make_environmental_shift_scenario(),
    make_clean_scenario("model_drift"),
    make_clean_scenario("catastrophic_forgetting"),
    make_clean_scenario("underfitting"),
    make_clean_scenario("overfitting"),
]

ALL_RESULTS = {}

for sc in SCENARIOS:
    print("\n" + "="*90)
    print("SCENARIO:", sc.name)

    if sc.name in ["baseline", "covariance_shift", "environmental_shift"]:
        met_xgb, met_lstm = evaluate_models(sc, lstm_clean, xgb_clean, tag=sc.name)
        xai = run_xai_for_scenario(sc, lstm_clean, xgb_clean)
        ALL_RESULTS[sc.name] = {"metrics":[met_xgb, met_lstm], "xai": xai, "meta": sc.meta}
        continue

    if sc.name == "model_drift":
        sc_eval, lstm_drift, xgb_drift = simulate_model_drift(sc)
        met_xgb, met_lstm = evaluate_models(sc_eval, lstm_drift, xgb_drift, tag=sc_eval.name)
        xai = run_xai_for_scenario(sc_eval, lstm_drift, xgb_drift)
        ALL_RESULTS[sc.name] = {"metrics":[met_xgb, met_lstm], "xai": xai, "meta": sc_eval.meta}
        continue

    if sc.name == "catastrophic_forgetting":
        (lstm_A, xgb_A), (lstm_B, xgb_B), scB = simulate_catastrophic_forgetting(sc)
        scA = make_clean_scenario("A_clean_eval")
        met_xgb_AA, met_lstm_AA = evaluate_models(scA, lstm_A, xgb_A, tag="A(on A)")
        met_xgb_BA, met_lstm_BA = evaluate_models(scA, lstm_B, xgb_B, tag="B(on A)")
        met_xgb_BB, met_lstm_BB = evaluate_models(scB, lstm_B, xgb_B, tag="B(on B)")
        xai_AA = run_xai_for_scenario(scA, lstm_A, xgb_A)
        xai_BA = run_xai_for_scenario(scA, lstm_B, xgb_B)
        xai_BB = run_xai_for_scenario(scB, lstm_B, xgb_B)
        ALL_RESULTS[sc.name] = {"metrics":[met_xgb_AA, met_lstm_AA, met_xgb_BA, met_lstm_BA, met_xgb_BB, met_lstm_BB],
                                "xai":{"A": xai_AA, "B_on_A": xai_BA, "B_on_B": xai_BB},
                                "meta":{"B_meta": scB.meta}}
        continue

    if sc.name == "underfitting":
        xgb_u = train_xgb_under_over(sc, "under")
        lstm_u = train_lstm_under_over(sc, "under")
        met_xgb, met_lstm = evaluate_models(sc, lstm_u, xgb_u, tag=sc.name)
        xai = run_xai_for_scenario(sc, lstm_u, xgb_u)
        ALL_RESULTS[sc.name] = {"metrics":[met_xgb, met_lstm], "xai": xai, "meta": sc.meta}
        continue

    if sc.name == "overfitting":
        xgb_o = train_xgb_under_over(sc, "over")
        lstm_o = train_lstm_under_over(sc, "over")
        met_xgb, met_lstm = evaluate_models(sc, lstm_o, xgb_o, tag=sc.name)
        xai = run_xai_for_scenario(sc, lstm_o, xgb_o)
        ALL_RESULTS[sc.name] = {"metrics":[met_xgb, met_lstm], "xai": xai, "meta": sc.meta}
        continue

SyntaxError: 'return' outside function (4083100640.py, line 239)

## 6) Ergebnisübersicht

In [ ]:
rows = []
for scen, payload in ALL_RESULTS.items():
    for m in payload["metrics"]:
        rows.append({"scenario": scen, **m})
metrics_df = pd.DataFrame(rows).sort_values(["scenario","name"]).reset_index(drop=True)
pd.options.display.float_format = "{:.6f}".format
metrics_df


In [ ]:
def print_xai_overview_for_entry(scenario_label: str, xai_dict: dict):
    print("\n" + "-"*90)
    print("XAI OVERVIEW –", scenario_label)

    ig = xai_dict.get("ig_lstm", {})
    print("\n[LSTM] Integrated Gradients:", ig.get("status"))
    if ig.get("status") == "ok":
        for f, v in ig.get("top_features", [])[:8]:
            print(f"  - {f}: {v:.6f}")
        pairs = ig.get("pairs", [])
        per_point = ig.get("per_point", {})
        if pairs:
            print("  Fixed Local Points:")
            for pid, ix in pairs:
                tops = per_point.get(pid, {}).get("top_features", [])[:5]
                tops_s = ", ".join([f"{f}({v:.3g})" for f, v in tops])
                print(f"    - pid={pid}, iloc={ix}: {tops_s}")

    ice = xai_dict.get("ice", {})
    print("\n[XGB-SEQ] ICE:", ice.get("status"), "| feature:", ice.get("feature"))
    if ice.get("status") == "ok" and ice.get("pairs"):
        print("  Fixed Local Points ICE:")
        for pid, ix in ice.get("pairs", []):
            print(f"    - pid={pid}, iloc={ix}")

    sur = xai_dict.get("surrogate_regression", {})
    print("\n[XGB-SEQ] Surrogate Ridge:", sur.get("status"),
          "| fidelity_R2_val:", sur.get("fidelity_R2_val"))

    cf = xai_dict.get("counterfactuals_rul", {})
    if cf:
        print("\n[XGB-SEQ] Counterfactuals (RUL):", cf.get("status"))
        ex = cf.get("examples", [])
        if ex:
            for e in ex[:3]:
                pid = e.get("point_id", None)
                ix  = e.get("iloc", e.get("index", None))
                print(f"  - pid={pid}, iloc={ix} | status={e.get('status')} | pred0={e.get('pred0')} -> pred_cf={e.get('pred_cf')}")


def print_all_xai_overview(ALL_RESULTS: dict):
    print("="*90)
    print("KOMPLETTÜBERSICHT: XAI Methoden pro Szenario & Modell")
    print("="*90)
    for scen, payload in ALL_RESULTS.items():
        xai = payload.get("xai", {})
        if scen != "catastrophic_forgetting":
            print_xai_overview_for_entry(scen, xai)
        else:
            for subkey in ["A", "B_on_A", "B_on_B"]:
                if subkey in xai:
                    print_xai_overview_for_entry(f"{scen}::{subkey}", xai[subkey])

print_all_xai_overview(ALL_RESULTS)